# Function 1 · Week 10 decision review

Narrow contamination source

[Detailed observations](official/exploration.ipynb) · [Detailed fits](official/model_predictions.ipynb) · [Previous pooled study](full-cohort/model_predictions.ipynb) · [2024 comparison](../../../side-quests/historical-cohort-2024/README.md) · [Course data-use review](../../../docs/capstone-data-policy.md)

**Decision status.** Week 10 is prepared and unsubmitted. Round 9 is the latest official return. Start with the [prepared comparison](#prepared-week-10-decision) when choosing between the two portal batches.

## Expectations

Round 9 tested a tight follow-up near the positive cluster. The archived GP and neural forecasts use signed-log height. The submitted choice came from a local distance-weighted comparison.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

from IPython.display import display
import numpy as np
import pandas as pd

FUNCTION = 1
WEEK_DIR = Path.cwd().resolve()
WEEK = int(WEEK_DIR.name.removeprefix("Week_"))
STAGE = WEEK_DIR.parents[2]
SHARED = STAGE / "notebooks/Shared"
SPECIAL = STAGE / "notebooks/Function_1/FunctionSpecificShared"
sys.path[:0] = [str(SHARED), str(SPECIAL)]
from weekly_peak_review import peak_review, portal_point
from query_budget import remaining_queries

analysis = WEEK_DIR / "official" if FUNCTION <= 4 else WEEK_DIR
official_path = analysis / "observations.csv"
model_snapshot = json.loads((analysis / "model-review.json").read_bytes())
observation_snapshot = json.loads((analysis / "observation-review.json").read_bytes())
feedback = json.loads((STAGE / f"submissions/round-{WEEK-1:02d}-results.json").read_bytes())
record = feedback["results"][FUNCTION - 1]
forecast = observation_snapshot["historical_forecast"]
if (model_snapshot["week"], model_snapshot["function"], model_snapshot["snapshot_sha256"]) != (
        WEEK, FUNCTION, sha256(official_path.read_bytes()).hexdigest()):
    raise ValueError("The official model review and observations describe different snapshots")
if (feedback["round"], record["function"], record["portal_input"], record["output"]) != (
        WEEK - 1, FUNCTION, forecast["portal_input"], forecast["actual_output"]):
    raise ValueError("The saved forecast must match the returned observation")
official = pd.read_csv(official_path, float_precision="round_trip")
cols = [c for c in official if c.startswith("x")]
x = official[cols].to_numpy(float)
raw_y = official.y.to_numpy(float)
later = official.source_round.to_numpy() > 0
best_row = official.loc[official.y.idxmax()]
incumbent = best_row[cols].to_numpy(float)
queries_left = remaining_queries(WEEK_DIR)
units = forecast["prediction_units"]
target = raw_y
if FUNCTION == 1:
    from function_1_model_helpers import signed_log
    target = signed_log(raw_y, scale=forecast["height_scale"])

display(pd.Series({
    "Archived forecast source": forecast["source"],
    "Submitted input": forecast["portal_input"],
    "Forecast units": units,
    "Saved GP mean": forecast["gp_mean"],
    "GP latent SD": forecast["gp_latent_sd"],
    "Saved neural mean": forecast["neural_mean"],
    "Neural seed SD": forecast["neural_seed_sd"],
}, name="Forecast frozen before the return"))

Archived forecast source    notebooks/Function_1/Week_09/official/figures/...
Submitted input                                             0.627434-0.628714
Forecast units                                            signed-log height h
Saved GP mean                                                      198.163301
GP latent SD                                                         11.29366
Saved neural mean                                                  130.822985
Neural seed SD                                                      17.007168
Name: Forecast frozen before the return, dtype: object

## Results

The third nearby positive return supports a narrow source. The 2024 near-2 return supports the same neighbourhood. Its presence does not bound a second source elsewhere.

In [2]:
latest_fitted = forecast["actual_in_prediction_units"]
display(pd.Series({"Returned score": record["output"], "Best before return": record["best_before"],
                   "Best now": record["best_after"], "Return rank": observation_snapshot["latest_rank"],
                   "Actual in forecast units": latest_fitted,
                   "Actual minus GP mean": forecast["actual_minus_gp_mean"],
                   "Actual minus neural mean": forecast["actual_minus_neural_mean"]}, name="Official result"))

# These references describe observed outcomes. The decision fits use official rows.
def optional_reference(folder, file, audit_name, expected):
    path = folder / file
    manifest = folder / audit_name
    if not (path.is_file() and manifest.is_file()):
        return None
    manifest_data = json.loads(manifest.read_bytes())
    if sha256(path.read_bytes()).hexdigest() != expected(manifest_data):
        raise ValueError(f"Reference hash mismatch: {path}")
    return pd.read_csv(path, float_precision="round_trip")

current_reference = optional_reference(
    STAGE / "data/cohort/2026-11-25", f"function_{FUNCTION}/observations.csv", "dataset-audit.json",
    lambda a: a["summary"][FUNCTION - 1]["sha256"],
)
archive = optional_reference(
    STAGE / "data/historical-cohort/2024", f"function_{FUNCTION}.csv", "intake-audit.json",
    lambda a: a["functions"][str(FUNCTION)]["sha256"],
)
reference_rows = []
if current_reference is not None:
    peer = current_reference.loc[current_reference.observation_id.str.startswith("peer-")]
    reference_rows.append(("Published current cohort", peer))
if archive is not None:
    reference_rows.append(("2024 archive, including extra starters", archive))
benchmarks = []
for label, frame in reference_rows:
    row = frame.loc[frame.y.idxmax()]
    benchmarks.append({"Scope": label, "Highest measured": row.y,
                       "Difference from your best": row.y - best_row.y,
                       "Distance to your best": np.linalg.norm(row[cols].to_numpy(float) - incumbent)})
if benchmarks:
    display(pd.DataFrame(benchmarks).round(8))
else:
    print("Local comparison snapshots are unavailable for this run.")

Returned score                1.993993
Best before return            1.997356
Best now                      1.997356
Return rank                   3.000000
Actual in forecast units    200.299724
Actual minus GP mean          2.136423
Actual minus neural mean     69.476738
Name: Official result, dtype: float64

,Scope,Highest measured,Difference from your best,Distance to your best
0,Published current cohort,1.979632,-0.017724,0.001961
1,"2024 archive, including extra starters",2.000000,0.002644,0.000742


In [3]:
from function_1_source_helpers import fit_gaussian_source, source_peak
source_models = []
for subset in (5, 6, 7):
    for anisotropic in (False, True):
        fit = fit_gaussian_source(x, raw_y, positive_count=subset, anisotropic=anisotropic)
        peak = source_peak(fit)
        source_models.append({"Shape": "Axis-aligned" if anisotropic else "Circular",
                              "Positive readings": subset, "Peak input": peak["portal_input"],
                              "Fitted raw peak": peak["predicted_height"],
                              "Residual degrees of freedom": fit["residual_degrees_of_freedom"]})
display(pd.DataFrame(source_models).round(7))

,Shape,Positive readings,Peak input,Fitted raw peak,Residual degrees of freedom
0,Circular,5,0.629424-0.627456,2.012341,1
1,Axis-aligned,5,0.629272-0.627660,2.008967,0
2,Circular,6,0.623963-0.632438,2.088331,2
3,Axis-aligned,6,0.628996-0.628060,2.003765,1
4,Circular,7,0.626598-0.630552,1.873379,3
5,Axis-aligned,7,0.629190-0.627878,2.006579,2


## Predictions

**Local peak and global evidence**

A local positive-source peak is likely. Compare the fitted peak locations and raw heights across the six source hypotheses. The full-domain signed-log GPs are stress tests for a narrow, irregular response. Their wide, unphysical extrapolation gives little basis for a global probability.

### Prepared Week 10 decision

**Prepared and unsubmitted as of 25 November 2026.** Round 9 is the latest confirmed feedback. Four portal queries remain per function, including Week 10. Use the cohort-informed route only if the course permits that use, including the exact published F2 repeat. The official-data fallback is ready if the rules remain unclear near the portal deadline.

The revised input uses the official seven-positive source fit constrained to the diagonal. The archived peak supports that constraint and its local fit predicts about 1.999997. The fallback uses the unconstrained official source peak. A plateau of near-2 returns would support the local source and leave distant peaks unresolved.

**Week 11 depends on the return.** Refine the better-supported local shape if the return improves. Use a distinct nearby control or a second-source test if it falls short.

[Week 10 course check and exact batches](../../../submissions/Week_10/README.md) · [Cohort advice and replay](../../../submissions/Week_10/advisory-review.md)

The estimates below use the advisory review's official-data GP comparator and predictive return standard deviation, which includes modelled observation noise. F8 uses the validated starter quadratic and omits a standard deviation because that fit supplies no probabilistic interval. The rows are proposed inputs with model estimates.

In [4]:
from week_10_decision_record import decision_comparison

with pd.option_context("display.max_colwidth", None):
    display(decision_comparison(STAGE, FUNCTION, official))

,Portal input,Official GP mean (score),Predictive return SD
Stage,,,
Initial proposal,0.629190-0.627878,2.007453,0.000303
Advisory experiment,0.629188-0.627801,2.007477,0.000338
Revised cohort-informed,0.628558-0.628558,2.004117,0.000038
Official-data fallback,0.629190-0.627878,2.007453,0.000303


### Conditional peak check

Two Gaussian processes with fitted noise share the same candidate set. A local candidate lies within the stated Euclidean radius of the observed incumbent. The columns report the fraction of joint latent draws in which none of the candidates in that set beats the incumbent by more than the working tolerance. The broader search includes Sobol points, corners, observed coordinates and surrogate-suggested probes. It cannot certify the continuous function or rule out unrepresented peaks. The nine grouped official return holdouts and nominal 90% observation-band counts indicate predictive fit. They do not calibrate the global-maximum event. The tolerance and radius are analysis choices.

In [5]:
radius = 0.02
tolerance = 0.001
peak = peak_review(x, target, later, radius=radius, tolerance=tolerance,
                   length_scales=[0.035, 0.035], noise=[0.04, 1e-06, 0.4],
                   minimum_distance=model_snapshot["minimum_query_distance"])
print(f"Local radius {radius:g}. Score tolerance {tolerance:g} in {units}.")
print(f"Joint posterior over {peak['candidate_count']} candidates, including {peak['local_count']} local."
      f" {peak['probability_draws']} draws per kernel.")
display(peak["summary"].round(5))
print("Adaptive two-query comparison on the Matérn shortlist")
one_query = peak["acquisitions"]["Matérn 3/2"].nlargest(1, "One-query EI")
print(f"Best one-query input: {one_query.iloc[0]['Input']}")
display(peak["acquisitions"]["Matérn 3/2"].head(5).round(6))

Local radius 0.02. Score tolerance 0.001 in signed-log height h.
Joint posterior over 438 candidates, including 141 local. 1024 draws per kernel.


,Local set within tolerance,Broader set within tolerance,Broader gap median,Broader gap 90th pct,Return MAE,Return RMSE,In 90% observation band,Return holdouts,Fitted noise SD,Fit warnings
Model,,,,,,,,,,
Matérn 3/2,0.00098,0.0,212.52631,296.10833,102.00966,143.39313,7,9,0.14294,1
RBF,0.00391,0.0,206.57354,284.66977,102.95904,143.19695,7,9,0.14294,1


Adaptive two-query comparison on the Matérn shortlist
Best one-query input: 0.695029-0.595581


,Input,Region,Mean,Observed SD,One-query EI,Two-query gain
0,0.695029-0.595581,Elsewhere,139.464344,116.129026,22.126377,43.281904
1,0.800506-0.616350,Elsewhere,138.464373,116.652641,22.009092,42.679935
5,0.678054-0.590139,Elsewhere,131.578464,120.627681,21.367449,42.409424
2,0.666517-0.756196,Elsewhere,138.328483,116.633641,21.962039,42.362654
4,0.611670-0.550514,Elsewhere,127.729678,123.901289,21.388421,42.209110


### Strategy for the remaining queries

Use the fitted positive-source shape for exploitation. A partitioning search such as DIRECT motivates a deliberate gap check if repeatable positives appear away from the main source. A short search can miss a narrow peak entirely.

The code compares one-query expected improvement in the returned score with a two-query approximation. It conditions on a first outcome, then chooses a different point on a shortlist. Both steps apply the saved minimum-spacing rule. It holds fitted kernel parameters fixed. Refit after each actual result and reassess the next action. The four rounds below are conditional decisions.

| Round | Action given the evidence so far |
| --- | --- |
| 10 | Compare the diagonal source refinement with the unconstrained official fit. Select the recorded route after checking assessed data use. |
| 11 | Refine the better-supported local shape if the return improves. Use a distinct nearby control or a second-source test if it falls short. |
| 12 | Use one away-from-peak probe only if the existing data suggest a second source that could beat the incumbent. Use the local cluster if that evidence stays weak. |
| 13 | Take the highest plausible local score consistent with the latest measurements. |

[Frazier, Bayesian optimisation tutorial](https://arxiv.org/abs/1807.02811) · [Jones, Perttunen and Stuckman, DIRECT](https://www.ressources-actuarielles.net/EXT/ISFA/1226.nsf/0/f9ca730ca27def69c12576d8002ed895/%24FILE/Jones93.pdf)

In [6]:
print(f"{queries_left} query opportunities per function, including Week {WEEK}.")
destination = WEEK_DIR / "weekly-peak-review.json"
summary = {"function": FUNCTION, "week": WEEK, "queries_remaining": queries_left,
           "official_snapshot_sha256": model_snapshot["snapshot_sha256"],
           "candidate_count": peak["candidate_count"], "local_count": peak["local_count"],
           "probability_draws": peak["probability_draws"], "local_radius": radius,
           "tolerance": tolerance, "target_units": units,
           "models": peak["summary"].reset_index().to_dict(orient="records"),
           "matérn_shortlist": peak["acquisitions"]["Matérn 3/2"].to_dict(orient="records")}
if FUNCTION == 8:
    summary["structural_fit"] = quadratic
destination.write_bytes((json.dumps(summary, indent=2, allow_nan=False) + "\n").encode())

4 query opportunities per function, including Week 10.


3614

### Course lifecycle

1. **Define the goal and scope** – choose the next official query while assessing how close the incumbent might be to the peak.
2. **Gather the data** – preserve the official snapshot, the frozen predictions and labelled published comparisons.
3. **Explore the data** – compare the returned result with local and external observations.
4. **Clean and preprocess the data** – verify hashes and portal alignment. Preserve the recorded raw score.
5. **Select and engineer features** – use the original coordinates, with a signed-log target for F1. Structural fits declare their terms separately.
6. **Define the machine-learning task** – regression supports maximisation and assessment of the incumbent.
7. **Partition the data** – withhold each official return and any matching portal coordinate for the predictive check.
8. **Select and train candidate methods** – compare two kernel assumptions and function-specific hypotheses.
9. **Evaluate and interpret the results** – judge forecast reliability and the finite-domain comparison before choosing how to spend the remaining budget.
10. **Deploy and iterate** – deployment is not applicable. The next portal return will update the review and the query plan.